In [12]:
# Basic libraries 
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pykalman import KalmanFilter
from itertools import combinations
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.stattools import coint
import statsmodels.api as sm
import pickle
import os
import sys

PROJECT_ROOT = os.path.abspath("..")
sys.path.insert(0, PROJECT_ROOT)

In [ ]:
# Scripts 
from src.cointegration import *
from src.kalman import *
from src.pairs import *
from src.trading_signal import *
from src.backtest import *
from src.plots import *
from src.utils import *
from src.refined_trading_signals import *

In [14]:
# Load in data from previous notebook
df1_is = pd.read_csv("/Users/ivanhung/Documents/GitHub/applied-project-06039211/data/df1_is",
                     index_col=0, parse_dates=True)
df1_oos = pd.read_csv("/Users/ivanhung/Documents/GitHub/applied-project-06039211/data/df1_oos",
                      index_col=0, parse_dates=True)

df2_is = pd.read_csv("/Users/ivanhung/Documents/GitHub/applied-project-06039211/data/df2_is",
                     index_col=0, parse_dates=True)
df2_oos = pd.read_csv("/Users/ivanhung/Documents/GitHub/applied-project-06039211/data/df2_oos",
                      index_col=0, parse_dates=True)

static_results_df = pd.read_csv("/Users/ivanhung/Documents/GitHub/applied-project-06039211/data/static_hedge_ratio")
dynamic_results_df = pd.read_csv("/Users/ivanhung/Documents/GitHub/applied-project-06039211/data/dynamic_hedge_ratio")

In [15]:
# Load dictionaries from previous notebook
with open("/Users/ivanhung/Documents/GitHub/applied-project-06039211/data/dictionaries/static_spreads.pkl", "rb") as f:
    static_spreads = pickle.load(f)
with open("/Users/ivanhung/Documents/GitHub/applied-project-06039211/data/dictionaries/static_models.pkl", "rb") as f:
    static_models = pickle.load(f)

with open("/Users/ivanhung/Documents/GitHub/applied-project-06039211/data/dictionaries/dynamic_spreads.pkl", "rb") as f:
    dyamic_spreads = pickle.load(f)
with open("/Users/ivanhung/Documents/GitHub/applied-project-06039211/data/dictionaries/dynamic_details.pkl", "rb") as f:
    dynamic_details = pickle.load(f)

with open("/Users/ivanhung/Documents/GitHub/applied-project-06039211/data/dictionaries/is_results.pkl", "rb") as f:
    is_results = pickle.load(f)
with open("/Users/ivanhung/Documents/GitHub/applied-project-06039211/data/dictionaries/is_trading_signals.pkl", "rb") as f:
    is_trading_signals = pickle.load(f)

with open("/Users/ivanhung/Documents/GitHub/applied-project-06039211/data/dictionaries/oos_results.pkl", "rb") as f:
    oos_results = pickle.load(f)
with open("/Users/ivanhung/Documents/GitHub/applied-project-06039211/data/dictionaries/oos_trading_signals.pkl", "rb") as f:
    oos_trading_signals = pickle.load(f)

In [16]:
# Load risk-free rate to calculate sharpe ratio
rf_is = pd.read_csv("/Users/ivanhung/Documents/GitHub/applied-project-06039211/data/rf_is",
                     index_col=0, parse_dates=True) 
rf_oos = pd.read_csv("/Users/ivanhung/Documents/GitHub/applied-project-06039211/data/rf_oos",
                     index_col=0, parse_dates=True)

# Refining trading signals to improve returns

Original trading strategy:

Mathematically, we can express it in a mapping:
$$
\text{Position}(z) =
\begin{cases}
-1 & \text{if } z > 2 \\
+1 & \text{if } z < -2 \\
\text{hold previous position} & \text{otherwise}
\end{cases}
$$